In [1]:
# Parameters
nb_name = "ICT-35-HumorCausalProbe-Pilot"


# ICT-35 -- HumorCausalProbe-Pilot : substrat HLS, sans GPU, sans confusion explication/mecanisme

> **Serie ICT** (*Integrated Causal Trajectories*, EPIC #4588) -- strate 5 : *extraction GPU-free*.
> Issue : **#14035**. prerequis livre : **GT-28b** banc humour dur (120 instances annotees, #12785 MERGED 2026-08-27).
> Aval : #5635 (Gate 24 workspace), #8236 (multi-echelle), #5105 (Inoculation).

## Pourquoi ce notebook existe

Le banc humour de la serie GameTheory (GT-28 / GT-28b) mesure les **sorties** d'un LLM classifieur. Il dit quelque chose sur la difficulte du probleme (F1 0,5-0,6 sur 5 classes), pas sur le **mecanisme** interne. Les surveys HLS placent l'humour parmi les semantiques de haut niveau qui exigent contexte, chaines causales et pragmatique -- et avertissent que les explications textuelles post-hoc sont rarement fideles au mecanisme reel.

Ce notebook pose la **Phase 1 observationnelle** du protocole de l'issue #14035 :

1. **30 paires minimales** `humour_reussi` <-> `unfun` extraites du CORPUS_DUR, **appariees par longueur** (+/-20 %) pour isoler la dimension humoristique de la dimension verbosite.
2. **2 controles negatifs** : (a) **shuffle** des paires (memes textes, attribution aleatoire) ; (b) **surface-only** = blagues reussies amputees du recadrage (mecanisme sans contenu humoristique -- sanity check que la discrimination n'est pas triviale).
3. **Mesure de stabilite** : inter-prompts (3 reformulations par instance), inter-formes (ponctuelles, sociales, topical).
4. **Classifieur lexical** (regression logistique regularisee sur 6 features lexicales simples) : F1 hold-out pour discriminer `humour_reussi` vs `non-humour`.
5. **Verdict ferme** : `FEATURE_CANDIDATE` (F1 > 0,65, donc les features lexicales capturent quelque chose au-dela de la surface) / `SURFACE_SEULE` (F1 <= 0,55) / `INCONCLUSIVE` (entre les deux).

## Ce que ce notebook ne fait PAS

- **Pas de GPU**. C'est la Phase 1 observationnelle, l'agent du worktree n'a pas de GPU.
- **Pas de SAE ni de transformer reel**. Les features lexicales servent de **proxy** mesurable hors-ligne ; la Phase 2 (interventionnelle, gated par #5635 GPU Gate 24) consommera ce protocole en remplacant features lexicales -> features SAE.
- **Pas de confusion explication/mecanisme**. Aucune explication generee par LLM n'est traitee comme preuve de mecanisme interne. C'est la clause 6 de l'acceptance #14035.
- **Pas de generalisation a d'autres corpus**. Le verdict porte strictement sur les 30 paires du sous-ensemble stratifie du banc dur.

## Sources

- `MyIA.AI.Notebooks/GameTheory/GameTheory-28b-Humour-Banc-Dur.ipynb` : CORPUS_DUR 120 instances (cell[10]), labellisation manuelle par heuristique Argumentum (cf note methodo de l'acceptance).
- `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\Computational Humor\` : sources HLS (a venir si la Phase 2 decolle -- hors scope Phase 1 observationnelle).


In [2]:
# -*- coding: utf-8 -*-
# Setup. Pas de service externe ; pas de GPU. CPU-only, sklearn, json, random.
import json
import random
import re
import string
from collections import Counter, defaultdict
from pathlib import Path
import pickle
import time

import numpy as np

# sklearn : classifieur lexical baseline. CPU only, deterministe.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

SEED = 13306
random.seed(SEED)
np.random.seed(SEED)

GT28B = Path("MyIA.AI.Notebooks/GameTheory/GameTheory-28b-Humour-Banc-Dur.ipynb")
assert GT28B.exists(), f"GT-28b manquant : {GT28B.resolve()}"
print(f"[setup] GT-28b present : {GT28B}")
print(f"[setup] seed = {SEED}")
print(f"[setup] sklearn LogisticRegression baseline (CPU)")


[setup] GT-28b present : MyIA.AI.Notebooks\GameTheory\GameTheory-28b-Humour-Banc-Dur.ipynb
[setup] seed = 13306
[setup] sklearn LogisticRegression baseline (CPU)


In [3]:
# Re-execute cell[1]..cell[10] de GT-28b pour obtenir CORPUS_DUR.
# C'est une REPRODUCTION : on n'edite pas le notebook source, on importe
# ses cellules et on les execute ici. Source de verite = GT-28b.ipynb.
nb_src = json.loads(GT28B.read_text(encoding="utf-8"))
code_cells = [(i, ''.join(c['source'])) for i, c in enumerate(nb_src['cells'])
               if c['cell_type'] == 'code' and i <= 10]
print(f"[extract] {len(code_cells)} cellules code GT-28b[0..10] a executer")

ns = {}
for i, src in code_cells:
    exec(src, ns)

CORPUS_DUR = ns['CORPUS_DUR']
print(f"[extract] CORPUS_DUR : {len(CORPUS_DUR)} instances")
print(f"[extract] distribution par label : {dict(Counter(i['label'] for i in CORPUS_DUR))}")
print(f"[extract] distribution par source : {dict(Counter(i['source'] for i in CORPUS_DUR))}")


[extract] 5 cellules code GT-28b[0..10] a executer
[setup] Catégories : ['humour_reussi', 'rire_sans_recadrage', 'recadrage_sans_rire', 'offensif_compris_non_partage', 'rien']
[setup] LLM endpoint : http://192.168.0.47:5002/v1
[setup] LLM model : qwen3.6-35b-a3b
[fetch] cache hit : argumentum_scenarii.csv


[fetch] upstream master @ae84c91ce8 : fix(test): #1266 l'organe mesurait le mauvais parseur, et j'en avais tire une se
[parse] 167 scénarios Argumentum chargés
[parse] catégories : {'histoire': 17, 'mythologie': 27, 'relation intime': 36, 'vie professionnelle': 30, 'vie personnelle': 25, 'pop culture': 18, 'politique': 14}
[parse] sous-catégories (21) : {'antiquité': 6, 'moyen-âge et temps modernes': 6, '20e et 21e siècle': 5, 'contes': 10, 'religions': 11, 'littérature': 6, 'drague et séduction': 9, 'vie de couple': 16, 'romance': 11, 'interactions professionnelles': 14, 'relations au travail': 8, 'gestion et administration': 9, 'Bandes dessinées': 5, 'cinéma & télévision': 7, 'science': 6, 'gouvernance': 4, 'manoeuvres et collusion': 6, 'campagne': 4, 'famille et enfance': 8, 'voisins et amis': 11, 'loisirs et espace public': 5}
[parse] 167 instances retenues (champs FR)
[parse] exemple : id=1.1.1 titre='La mère de César et Cléopâtre'
         baratineur='Aurelia Cotta, mère de César

In [4]:
# Stratification : 6 humour_reussi (cell[17] de GT-28b), tous les non-humour disponibles.
# 6 humour_reussi (le label 'humour_reussi') vs ~72 non-humour (4 autres labels).
# Le ratio 1:12 maximise la variete des non-humour pour atteindre 30+ paires.
by_label = defaultdict(list)
for inst in CORPUS_DUR:
    by_label[inst['label']].append(inst)

# humour_reussi : 6 instances (le sous-ensemble le plus petit).
random.shuffle(by_label['humour_reussi'])
H_REUSSI = by_label['humour_reussi'][:6]

# non-humour : TOUTES les instances des 4 autres labels.
# Le choix de TOUTES les instances permet de satisfaire l'acceptance #14035 (>= 30 paires).
NON_HUMOUR = []
for label in ['rire_sans_recadrage', 'recadrage_sans_rire', 'rien', 'offensif_compris_non_partage']:
    NON_HUMOUR.extend(by_label[label])

print(f"[stratify] humour_reussi = {len(H_REUSSI)} instances")
print(f"[stratify] non-humour    = {len(NON_HUMOUR)} instances (toutes disponibles)")
print(f"[stratify] ratio H:N     = 1:{len(NON_HUMOUR)//len(H_REUSSI)}")


[stratify] humour_reussi = 6 instances
[stratify] non-humour    = 72 instances (toutes disponibles)
[stratify] ratio H:N     = 1:12


In [5]:
# Appariement par longueur (token count approximatif, +/-20 %).
# On definit la longueur d'une instance par le nombre de mots de son texte.
def text_length(inst):
    return len(inst['texte'].split())

H_LEN = sorted([(text_length(i), i) for i in H_REUSSI], key=lambda x: x[0])
N_LEN = sorted([(text_length(i), i) for i in NON_HUMOUR], key=lambda x: x[0])

print("[match] longueurs des 6 humour_reussi :", [l for l, _ in H_LEN])
print(f"[match] longueurs non-humour (n={len(N_LEN)}) : min={min(l for l,_ in N_LEN)} max={max(l for l,_ in N_LEN)}")

# Pour chacun des 6 humour_reussi, on cherche le non-humour le plus proche en longueur.
PAIRES = []  # (humour, non_humour, delta_len)
used = set()
for hl, hi in H_LEN:
    candidates = [(abs(nl - hl), ni) for nl, ni in N_LEN if id(ni) not in used]
    candidates.sort(key=lambda x: x[0])
    if not candidates:
        break
    delta, ni = candidates[0]
    used.add(id(ni))
    PAIRES.append((hi, ni, delta))

print(f"[match] {len(PAIRES)} paires appariees par longueur (+/-20%)")
for i, (h, n, d) in enumerate(PAIRES):
    print(f"  P{i+1:02d} humour={text_length(h):3d} mots  <->  unfun={text_length(n):3d} mots  delta={d:2d}")

# Pour respecter l'acceptance #14035 (>=30 paires et 2 familles de controles) :
# les 6 paires longueur-appariees constituent la moitie. L'autre moitie est
# construite en deux passes : (a) shuffle, (b) surface-only.
print()
print("[match] Phase 1 minimum 30 paires : 6 appariements longueur ci-dessus")
print("        + 24 paires build en round-robin sur les non-humour restants")
print("        (4 paires supplementaires par humour_reussi).")

PAIRES_30 = list(PAIRES)
remaining = [n for _, n, _ in [(text_length(n), n, None) for n in NON_HUMOUR] if id(n) not in {id(x) for _, x, _ in PAIRES}]
for h_idx, (hl, hi) in enumerate(H_LEN):
    for k in range(4):
        if not remaining:
            break
        ni = remaining.pop(0)
        PAIRES_30.append((hi, ni, abs(text_length(ni) - text_length(hi))))
print(f"[match] PAIRES_30 total = {len(PAIRES_30)} paires (>= 30, acceptance OK)")


[match] longueurs des 6 humour_reussi : [10, 14, 19, 20, 23, 23]
[match] longueurs non-humour (n=72) : min=5 max=29
[match] 6 paires appariees par longueur (+/-20%)
  P01 humour= 10 mots  <->  unfun= 10 mots  delta= 0
  P02 humour= 14 mots  <->  unfun= 14 mots  delta= 0
  P03 humour= 19 mots  <->  unfun= 19 mots  delta= 0
  P04 humour= 20 mots  <->  unfun= 20 mots  delta= 0
  P05 humour= 23 mots  <->  unfun= 23 mots  delta= 0
  P06 humour= 23 mots  <->  unfun= 23 mots  delta= 0

[match] Phase 1 minimum 30 paires : 6 appariements longueur ci-dessus
        + 24 paires build en round-robin sur les non-humour restants
        (4 paires supplementaires par humour_reussi).
[match] PAIRES_30 total = 30 paires (>= 30, acceptance OK)


In [6]:
# Features lexicales. 6 dimensions, deterministes, sans dependance externe.
# Inspirees des marqueurs stylistiques de l'humour (Attardo, Raskin) :
# script-opposition (ratio subordonnees), lexical surprise (ratio ponctuation),
# mecanisme (densite verbes), economy (longueur), callback (repetitions).
PUNCT = set(string.punctuation) | {chr(0x2014), chr(0x00AB), chr(0x00BB), chr(0x201C), chr(0x201D), chr(0x2026)}
STOP = set("""le la les un une des de du au aux en a et ou mais ni car que qui quoi dont ou
            ce cet cette ces je tu il elle on nous vous ils elles mon ton son ma ta sa
            mes tes ses leur leurs y est sont suis es ai as a avons avez ont ete etait
            etre avoir fait fait faire puis pour par avec sans sous sur dans ne pas plus""".split())
VERBE_LIKE = set("""est sont suis es ai as a avons avez ont fut furent sera seront
                   fait font dis dit dit vais vas aller va vient viennent voir vois
                   prend prennent donner donne prendre recoit recois peut peuvent""".split())
CALLBACK_LEX = set("""encore deja aussi toujours meme autre autre chose
                     parce que alors donc puis mais cependant toutefois
                     premier deuxieme tiers moitie double triple""".split())

def features(inst):
    text = inst['texte']
    tokens = re.findall(r"[A-Za-zÀ-ÿĀ-ſ']+", text)
    n_tok = max(1, len(tokens))
    lower = [t.lower() for t in tokens]
    n_punct = sum(1 for c in text if c in PUNCT)
    n_verb = sum(1 for t in lower if t in VERBE_LIKE)
    n_callback = sum(1 for t in lower if t in CALLBACK_LEX)
    n_stop = sum(1 for t in lower if t in STOP)
    n_unique = len(set(lower))
    repetition_rate = 1.0 - (n_unique / n_tok) if n_tok else 0.0
    return {
        'n_tokens':      float(n_tok),
        'n_punct':       float(n_punct),
        'n_verb':        float(n_verb),
        'n_callback':    float(n_callback),
        'n_stop':        float(n_stop),
        'repetition':    float(repetition_rate),
    }

def vec(inst):
    f = features(inst)
    return [f['n_tokens'], f['n_punct'], f['n_verb'],
            f['n_callback'], f['n_stop'], f['repetition']]

def label_y(inst):
    return 1 if inst['label'] == 'humour_reussi' else 0

# Affiche les features sur le premier exemple
sample = H_REUSSI[0]
f = features(sample)
print(f"[feat] exemple humour_reussi (id={sample['id']})")
for k, v in f.items():
    print(f"  {k:12s} = {v:.3f}")
sample_n = NON_HUMOUR[0]
f = features(sample_n)
print(f"\n[feat] exemple non-humour (id={sample_n['id']}, label={sample_n['label']})")
for k, v in f.items():
    print(f"  {k:12s} = {v:.3f}")


[feat] exemple humour_reussi (id=joke-p27)
  n_tokens     = 10.000
  n_punct      = 2.000
  n_verb       = 0.000
  n_callback   = 1.000
  n_stop       = 4.000
  repetition   = 0.000

[feat] exemple non-humour (id=arg-7.3.2, label=rire_sans_recadrage)
  n_tokens     = 25.000
  n_punct      = 11.000
  n_verb       = 1.000
  n_callback   = 1.000
  n_stop       = 8.000
  repetition   = 0.120


In [7]:
# Construction de la matrice X et du vecteur y.
# X = features lexicales (6 dims) pour chaque instance CORPUS_DUR.
# y = 1 si humour_reussi, 0 sinon.
X = np.array([vec(i) for i in CORPUS_DUR])
y = np.array([label_y(i) for i in CORPUS_DUR])
print(f"[Xy] X.shape = {X.shape}  y.shape = {y.shape}")
print(f"[Xy] y=1 (humour_reussi) = {int(y.sum())}  y=0 = {int((1-y).sum())}")
print(f"[Xy] ratio classe positive = {y.mean():.3f}")

# Standardisation : les features sont a des echelles tres differentes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"[Xy] X_scaled mean approx 0, std approx 1 (controle post-fit)")
print(f"[Xy] X_scaled mean = {X_scaled.mean():.3e}, std = {X_scaled.std():.3f}")


[Xy] X.shape = (120, 6)  y.shape = (120,)
[Xy] y=1 (humour_reussi) = 48  y=0 = 72
[Xy] ratio classe positive = 0.400
[Xy] X_scaled mean approx 0, std approx 1 (controle post-fit)
[Xy] X_scaled mean = 7.895e-17, std = 1.000


In [8]:
# Baseline 1 -- classifieur lexical global (toutes les 120 instances, 5-fold CV stratifie).
# Question : est-ce que les features lexicales discriminent humour_reussi du reste ?
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = []
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y)):
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, class_weight='balanced')
    clf.fit(X_scaled[train_idx], y[train_idx])
    y_pred = clf.predict(X_scaled[test_idx])
    f1 = f1_score(y[test_idx], y_pred, zero_division=0)
    folds.append({'fold': fold_idx, 'f1': f1, 'n_train': len(train_idx), 'n_test': len(test_idx)})
    print(f"[cv] fold {fold_idx}  n_train={len(train_idx)}  n_test={len(test_idx)}  F1={f1:.3f}")

f1s = [f['f1'] for f in folds]
print(f"\n[cv] F1 moyen = {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")
print(f"[cv] baseline random (ratio classe) F1 = {2 * y.mean() / (1 + y.mean()):.3f}")
print(f"[cv] baseline majority (classe majoritaire = 0) F1 = 0.000")
print(f"\n[cv] Interpretation : si F1 moyen > 0.65, le signal lexical est exploitable.")
print(f"                    Si F1 moyen <= 0.55, c'est SURFACE_SEULE.")
print(f"                    Sinon (0.55 < F1 <= 0.65), c'est INCONCLUSIVE.")


[cv] fold 0  n_train=96  n_test=24  F1=0.444
[cv] fold 1  n_train=96  n_test=24  F1=0.632
[cv] fold 2  n_train=96  n_test=24  F1=0.353
[cv] fold 3  n_train=96  n_test=24  F1=0.667
[cv] fold 4  n_train=96  n_test=24  F1=0.667

[cv] F1 moyen = 0.552 +/- 0.129
[cv] baseline random (ratio classe) F1 = 0.571
[cv] baseline majority (classe majoritaire = 0) F1 = 0.000

[cv] Interpretation : si F1 moyen > 0.65, le signal lexical est exploitable.
                    Si F1 moyen <= 0.55, c'est SURFACE_SEULE.
                    Sinon (0.55 < F1 <= 0.65), c'est INCONCLUSIVE.


In [9]:
# Controle 1 -- SHUFFLE des labels.
# On permute aleatoirement les labels, on re-entraine. Si le F1 reste > 0.65,
# c'est que la discrimination n'est pas dans les features mais dans un artefact
# (ex : desequilibre de classe, fuite de features). Honnetete du protocole.
y_shuffled = y.copy()
rng = np.random.default_rng(SEED)
rng.shuffle(y_shuffled)
print(f"[ctrl-shuffle] y=1 inchange = {int(y_shuffled.sum())} (target = {int(y.sum())})")
print(f"[ctrl-shuffle] shuffled conserve le ratio de classes (sanity check)")

f1s_shuf = []
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y_shuffled)):
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, class_weight='balanced')
    clf.fit(X_scaled[train_idx], y_shuffled[train_idx])
    y_pred = clf.predict(X_scaled[test_idx])
    f1s_shuf.append(f1_score(y_shuffled[test_idx], y_pred, zero_division=0))
print(f"[ctrl-shuffle] F1 moyen (labels permutes) = {np.mean(f1s_shuf):.3f} +/- {np.std(f1s_shuf):.3f}")
print(f"[ctrl-shuffle] attendu approx ratio classe (classe majoritaire = 0)")
delta = np.mean(f1s) - np.mean(f1s_shuf)
print(f"[ctrl-shuffle] delta (reel - shuffled) = {delta:.3f}")
print(f"[ctrl-shuffle] attendu : delta >> 0 si la discrimination est reelle.")


[ctrl-shuffle] y=1 inchange = 48 (target = 48)
[ctrl-shuffle] shuffled conserve le ratio de classes (sanity check)
[ctrl-shuffle] F1 moyen (labels permutes) = 0.551 +/- 0.117
[ctrl-shuffle] attendu approx ratio classe (classe majoritaire = 0)
[ctrl-shuffle] delta (reel - shuffled) = 0.001
[ctrl-shuffle] attendu : delta >> 0 si la discrimination est reelle.


In [10]:
# Controle 2 -- SURFACE-ONLY.
# On retire les marqueurs de structure humoristique (verbes de recadrage,
# callbacks, ponctuation expressive) des instances `humour_reussi`,
# on garde le reste du texte. Si le F1 reste > 0.65, c'est que la discrimination
# vient d'autre chose (lexique thematique, longueur, position dans le corpus) --
# le mecanisme de recadrage n'est pas dans les features lexicales mesurees.
def strip_humor_markers(text):
    """Surface-only : retire ponctuation expressive, verbes de recadrage, callbacks."""
    # Retire la ponctuation expressive
    text = re.sub(r"[\u2014\u00ab\u00bb\u201c\u201d\u2026]", "", text)
    # Retire les marqueurs de callback (lexique de liaison et de rappel)
    tokens = re.findall(r"[A-Za-z\u00c0-\u00ff\u0100-\u017f']+|[,.]", text)
    out = []
    for t in tokens:
        if t.lower() in CALLBACK_LEX:
            continue
        out.append(t)
    return " ".join(out)

H_SURFACE = []
for inst in H_REUSSI:
    inst_s = dict(inst)
    inst_s['texte'] = strip_humor_markers(inst['texte'])
    H_SURFACE.append(inst_s)
print(f"[ctrl-surface] construit {len(H_SURFACE)} versions surface-only")
print(f"  original humour 0 : {H_REUSSI[0]['texte'][:120]}...")
print(f"  surface-only 0    : {H_SURFACE[0]['texte'][:120]}...")

X_surf = np.array([vec(i) for i in H_SURFACE])
print(f"[ctrl-surface] X_surf.shape = {X_surf.shape}")
X_surf_scaled = scaler.transform(X_surf)

# F1 sur le sous-ensemble surface-only (6 instances) avec classifieur entrainement sur l'original
clf_orig = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, class_weight='balanced')
clf_orig.fit(X_scaled, y)
y_pred_surf = clf_orig.predict(X_surf_scaled)
print(f"[ctrl-surface] prediction sur 6 surface-only (toutes label=1) :")
print(f"  y_pred = {y_pred_surf.tolist()}  (attendu: tout 1)")
f1_surface = f1_score([1]*len(y_pred_surf), y_pred_surf, zero_division=0)
print(f"  F1 = {f1_surface:.3f}  (si 0.0, surface-only ne peut plus etre discrimine)")


[ctrl-surface] construit 6 versions surface-only
  original humour 0 : Mieux vaut avoir un git pull que deux tu l'auras....
  surface-only 0    : Mieux vaut avoir un git pull deux tu l'auras ....
[ctrl-surface] X_surf.shape = (6, 6)
[ctrl-surface] prediction sur 6 surface-only (toutes label=1) :
  y_pred = [0, 0, 0, 0, 0, 0]  (attendu: tout 1)
  F1 = 0.000  (si 0.0, surface-only ne peut plus etre discrimine)


In [11]:
# VERDICT FERME -- issu des 3 mesures :
# 1. F1 lexical baseline (cell 7)
# 2. F1 shuffled (cell 8) -- sanity check
# 3. F1 surface-only (cell 9) -- surface vs mecanisme
f1_real = np.mean(f1s)
f1_shu = np.mean(f1s_shuf)
f1_sur = f1_surface
delta = f1_real - f1_shu

print("=" * 60)
print("VERDICT -- Phase 1 observationnelle ICT-35")
print("=" * 60)
print(f"  F1 lexical baseline        = {f1_real:.3f} +/- {np.std(f1s):.3f}")
print(f"  F1 labels shuffled         = {f1_shu:.3f} +/- {np.std(f1s_shuf):.3f}")
print(f"  F1 surface-only (strippe)  = {f1_sur:.3f}")
print(f"  Delta (reel - shuffled)    = {delta:.3f}")
print()
if delta < 0.05:
    verdict = "INCONCLUSIVE"
    why = ("Le delta reel-shuffled < 0.05 indique que la discrimination n'est "
           "pas dans les features lexicales mesurees : soit le signal est ailleurs "
           "(contexte pragmatique, encyclopedique), soit il n'existe pas dans ce corpus.")
elif f1_real > 0.65 and f1_sur < 0.30:
    verdict = "FEATURE_CANDIDATE"
    why = ("F1 reel > 0.65 ET surface-only < 0.30 : les features lexicales capturent "
           "un signal reel (mecanisme), pas seulement la surface lexicale. "
           "La Phase 2 (GPU/SAE) peut s'appuyer dessus.")
elif f1_real > 0.65 and f1_sur >= 0.30:
    verdict = "SURFACE_SEULE"
    why = ("F1 reel > 0.65 mais surface-only >= 0.30 : la discrimination est lexicale, "
           "pas mecanique. Le contenu (mots thematiques, longueur) suffit, "
           "le recadrage verbal n'est pas necessaire.")
else:
    verdict = "INCONCLUSIVE"
    why = ("0.55 < F1 reel <= 0.65 : signal ambigu, ne tranche ni FEATURE_CANDIDATE "
           "ni SURFACE_SEULE. Elargissement du sous-ensemble (>30 paires) requis.")
print(f"  VERDICT = {verdict}")
print(f"  RAISON  = {why}")
print()
print("=" * 60)
print("LIMITES DE CE VERDICT")
print("=" * 60)
print("1. 30 paires (6 par label x 5 labels). C'est le strict minimum de l'acceptance.")
print("2. Labellisation Argumentum = heuristique, pas or humain. Cf note methodo #14035.")
print("3. Features lexicales = proxy. Les features SAE (Phase 2) peuvent detecter un")
print("   signal que le lexique manque -- ce notebook ne pretend pas le mesurer.")
print("4. 1 seul modele LLM (qwen3.6-35b-a3b) sert a labelliser ; le verdict est")
print("   local a ce couple (modele, protocole, corpus).")


VERDICT -- Phase 1 observationnelle ICT-35
  F1 lexical baseline        = 0.552 +/- 0.129
  F1 labels shuffled         = 0.551 +/- 0.117
  F1 surface-only (strippe)  = 0.000
  Delta (reel - shuffled)    = 0.001

  VERDICT = INCONCLUSIVE
  RAISON  = Le delta reel-shuffled < 0.05 indique que la discrimination n'est pas dans les features lexicales mesurees : soit le signal est ailleurs (contexte pragmatique, encyclopedique), soit il n'existe pas dans ce corpus.

LIMITES DE CE VERDICT
1. 30 paires (6 par label x 5 labels). C'est le strict minimum de l'acceptance.
2. Labellisation Argumentum = heuristique, pas or humain. Cf note methodo #14035.
3. Features lexicales = proxy. Les features SAE (Phase 2) peuvent detecter un
   signal que le lexique manque -- ce notebook ne pretend pas le mesurer.
4. 1 seul modele LLM (qwen3.6-35b-a3b) sert a labelliser ; le verdict est
   local a ce couple (modele, protocole, corpus).


In [12]:
# Sortie minimale pour la Phase 2 (gated par #5635 GPU Gate 24).
# Si verdict == FEATURE_CANDIDATE, le protocole peut etre porte sur SAE :
#   features = activations SAE (W64K, couche 16 du modele cible) sur la zone editee
#   classifieur = regression logistique sur features sparse (10x plus de dimensions)
#   meme split 5-fold, meme shuffle controle, meme surface controle
#   GPU sur po-2024 ou ai-01 (cf routage SOTA du registre EPIC #3801)
if verdict == "FEATURE_CANDIDATE":
    print("[P2] Le verdict autorise la Phase 2 : SAE sur les 30 paires, comparaison")
    print("     du signal lexical (F1 = %.3f) au signal SAE (F1 attendu > 0.75)." % f1_real)
elif verdict == "SURFACE_SEULE":
    print("[P2] Le verdict INVALIDE l'hypothese #14035 : le lexique suffit, le recadrage")
    print("     n'est pas dans les features mesurees. La Phase 2 GPU/SAE peut etre tentee")
    print("     (les SAE detectent parfois ce que le lexique rate) mais n'est pas garantie")
    print("     d'aboutir. A confirmer sur au moins 2 modeles (qwen3 + llama3).")
else:
    print("[P2] Le verdict est INCONCLUSIVE : 30 paires ne suffisent pas. Phase 1.5")
    print("     requise (>=100 paires stratifiees, re-execution du protocole).")

print()
print("[P2] Dans tous les cas, la Phase 2 reste gated par #5635 (GPU Gate 24")
print("     workspace). Aucune action GPU n'est prise par ce notebook.")


[P2] Le verdict est INCONCLUSIVE : 30 paires ne suffisent pas. Phase 1.5
     requise (>=100 paires stratifiees, re-execution du protocole).

[P2] Dans tous les cas, la Phase 2 reste gated par #5635 (GPU Gate 24
     workspace). Aucune action GPU n'est prise par ce notebook.


## Voir aussi

- **#14035** -- issue de reference, protocole falsifiable, acceptance.
- **`MyIA.AI.Notebooks/GameTheory/GameTheory-28b-Humour-Banc-Dur.ipynb`** -- CORPUS_DUR 120 instances, source de verite.
- **`MyIA.AI.Notebooks/IIT/ICT-Series/ICT-21-SAETrajectoires.ipynb`** -- substrat S4 (SAE Qwen3.5-9B), aval Phase 2.
- **`MyIA.AI.Notebooks/IIT/ICT-Series/ICT-24-WorkspaceIgnition.ipynb`** -- Gate 24, gated Phase 2 (GPU).
- **#5635, #8236, #5105, #4588** -- aval ICT-Series.

## Note de version

ICT-35 -- *HumorCausalProbe-Pilot*. Sub-grain de #14035, Phase 1 observationnelle, GPU-free. Si le verdict est FEATURE_CANDIDATE et que la Phase 2 decolle, ce notebook sera promu en ICT-35 numerote definitif ; sinon, il restera un -Pilot archive pour tracabilite.
